In [29]:
import nltk
import re
nltk.download('rte')
nltk.download('stopwords')
from nltk.corpus import stopwords
import pickle

[nltk_data] Downloading package rte to /Users/hrbpkumar/nltk_data...
[nltk_data]   Package rte is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/hrbpkumar/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [34]:
def get_word_features(review):
    stemmer = nltk.stem.PorterStemmer()
    words = [word if (word[0:2] == '__') else word.lower() \
                     for word in review.split() \
                     if len(review) >= 3]
    stemmed_words = [stemmer.stem(w) for w in words] 
    print(stemmed_words)

    def get_word_features(words):
            bag = {}
            stop_words = set(stopwords.words('english'))
            filtered_words = [w for w in words if not w in stop_words]
            words_uni = ['has(%s)' % ug for ug in filtered_words]
            for f in words_uni:
                bag[f] = 1

            # bag = collections.Counter(words_uni+words_bi+words_tri)
            return bag

    negtn_regex = re.compile(r"""(?:
            ^(?:never|no|nothing|nowhere|noone|none|not|
                havent|hasnt|hadnt|cant|couldnt|shouldnt|
                wont|wouldnt|dont|doesnt|didnt|isnt|arent|aint
            )$
          )
          |
          n't
          """, re.X)

    pos_regex = re.compile(r"""(?:
                    ^(?:excellent|wow|awesome|happy|cool|good|love|
                        wonderful|amazing|amaze|bliss|enjoy|fantastic|
                        beautiful|beauty|better|very good|fun|funny|arent|luck|lucky|
                        nice|super|great
                    )$
                )
                |
                n't
                """, re.X)

    def get_negation_features(words):
            INF = 0.0
            negtn = [bool(negtn_regex.search(w)) for w in words]

            left = [0.0] * len(words)
            prev = 0.0
            for i in range(0, len(words)):
                if (negtn[i]):
                    prev = 1.0
                left[i] = prev
                prev = max(0.0, prev - 0.1)

            right = [0.0] * len(words)
            prev = 0.0
            for i in reversed(range(0, len(words))):
                if (negtn[i]):
                    prev = 1.0
                right[i] = prev
                prev = max(0.0, prev - 0.1)

            return dict(zip(
                ['neg_l(' + w + ')' for w in words] + ['neg_r(' + w + ')' for w in words],
                left + right))

    def get_positive_features(words):

            bag={}
            for word in words:
                if bool(pos_regex.search(word)):
                    key = 'pos(' + word + ')'
                    bag[key] = 1
            return bag


    def extract_features(words):

            features = {}
            negation_features = get_negation_features(words)
            features.update(negation_features)
            postive_features = get_positive_features(words)
            features.update(postive_features)
            word_features = get_word_features(words)
            features.update(word_features)
            #sys.stderr.write('\rfeatures extracted for ' + str(extract_features.count) + ' reviews')
            return features

    word_features = extract_features(stemmed_words)
    print(word_features)
    return word_features

    

In [42]:
get_word_features("great value for money")

['great', 'valu', 'for', 'money']
{'neg_l(great)': 0.0, 'neg_l(valu)': 0.0, 'neg_l(for)': 0.0, 'neg_l(money)': 0.0, 'neg_r(great)': 0.0, 'neg_r(valu)': 0.0, 'neg_r(for)': 0.0, 'neg_r(money)': 0.0, 'pos(great)': 1, 'has(great)': 1, 'has(valu)': 1, 'has(money)': 1}


{'neg_l(great)': 0.0,
 'neg_l(valu)': 0.0,
 'neg_l(for)': 0.0,
 'neg_l(money)': 0.0,
 'neg_r(great)': 0.0,
 'neg_r(valu)': 0.0,
 'neg_r(for)': 0.0,
 'neg_r(money)': 0.0,
 'pos(great)': 1,
 'has(great)': 1,
 'has(valu)': 1,
 'has(money)': 1}

In [36]:


# Load the saved model file
with open("./models/NaiveBayesClassifier_pkl_model.pkl", "rb") as file:
    loaded_model = pickle.load(file)

['the', 'prodict', 'is', 'awesom', 'and', 'veri', 'nice']
{'neg_l(the)': 0.0, 'neg_l(prodict)': 0.0, 'neg_l(is)': 0.0, 'neg_l(awesom)': 0.0, 'neg_l(and)': 0.0, 'neg_l(veri)': 0.0, 'neg_l(nice)': 0.0, 'neg_r(the)': 0.0, 'neg_r(prodict)': 0.0, 'neg_r(is)': 0.0, 'neg_r(awesom)': 0.0, 'neg_r(and)': 0.0, 'neg_r(veri)': 0.0, 'neg_r(nice)': 0.0, 'pos(nice)': 1, 'has(prodict)': 1, 'has(awesom)': 1, 'has(veri)': 1, 'has(nice)': 1}
Predicted Sentiment: negative


In [52]:
# Use it to predict new data
# text_features must be in the exact same format (dictionary) your model trained on
prediction = loaded_model.classify(get_word_features("The product is great value"))
print("Predicted Sentiment:", prediction)

['the', 'product', 'is', 'great', 'valu']
{'neg_l(the)': 0.0, 'neg_l(product)': 0.0, 'neg_l(is)': 0.0, 'neg_l(great)': 0.0, 'neg_l(valu)': 0.0, 'neg_r(the)': 0.0, 'neg_r(product)': 0.0, 'neg_r(is)': 0.0, 'neg_r(great)': 0.0, 'neg_r(valu)': 0.0, 'pos(great)': 1, 'has(product)': 1, 'has(great)': 1, 'has(valu)': 1}
Predicted Sentiment: positive
